# Fine-tune Mira's LLM on persona-chat (Colab → local Ollama)

Trains **Qwen2.5-3B-Instruct** on the
[`Cynaptics/persona-chat`](https://huggingface.co/datasets/Cynaptics/persona-chat)
dataset with a 4-bit QLoRA, then exports a **GGUF + Ollama Modelfile** you download
and run on your own machine. We use the 3B (not 7B) on purpose: it trains ~2× faster
so it finishes comfortably inside a free Colab session, and once it's your Ollama
model it runs well on your 6 GB GTX 1660 alongside Whisper + TTS — the size the main
README calls the 6 GB sweet spot.

**Sized for free Colab.** The defaults (`MAX_SEQ_LEN=1024`, `MAX_ROWS=8000`) finish
in well under an hour on a free T4. Checkpoints are saved to **Google Drive** so if
Colab cuts you off, you just Run-all again and it **resumes** where it left off.

**What it teaches:** each row's *own* persona becomes the system prompt and the
gold reply is the target, and we train on the responses only. So the model learns
the transferable skill — *follow the persona in the system prompt and reply like a
person* — not one baked-in character. Mira's real persona keeps getting injected at
runtime by `prefrontal_cortex.py`, so this just sharpens conversational quality.

### How to run
1. **Runtime → Change runtime type → GPU** (T4 is fine, and free).
2. **Runtime → Run all.** Early on it asks to mount Google Drive — click through
   the auth popup once (that's where checkpoints + the final model are saved).
3. The last cell downloads `mira-gguf.zip` (also saved to your Drive). Unzip it on
   your PC, then locally:
   ```
   cd mira-gguf
   ollama create mira -f Modelfile
   ```
   and point Mira at it: `MIRA_MODEL=mira` (PowerShell: `$env:MIRA_MODEL = "mira"`).

## 1. Check the GPU and install Unsloth

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
# Unsloth pulls a matching torch/transformers/trl/peft/bitsandbytes set for Colab.
# If `import unsloth` later fails, grab the current install snippet from
# https://docs.unsloth.ai/get-started/installing-+-updating
!pip install unsloth

## 2. Config

Everything you'd want to tweak lives here. The defaults are sized to finish on a
free T4 in well under an hour.

- **Want maximum quality and have Colab Pro (or patience to resume across sessions)?**
  Set `BASE_MODEL = "unsloth/Qwen2.5-7B-Instruct"` and/or `MAX_ROWS = 0`. The 7B is a
  better model but trains ~2× slower and is a tight fit running locally on 6 GB.
- **Use the whole dataset / longer context:** `MAX_ROWS = 0`, `MAX_SEQ_LEN = 2048`
  (longer runs; lean on the Drive checkpoints to resume if you get cut off).

In [ ]:
BASE_MODEL        = "unsloth/Qwen2.5-3B-Instruct"   # ~2x faster than 7B, fits your 6 GB card well
DATASET           = "Cynaptics/persona-chat"
OLLAMA_MODEL_NAME = "mira"            # the `ollama create <name>` tag -> your MIRA_MODEL
GGUF_QUANT        = "q4_k_m"          # matches Ollama's stock qwen2.5 quant

MAX_SEQ_LEN   = 1024                  # persona-chat turns are short; 1024 covers them,
                                     #   ~halves time, and avoids the T4 VRAM-offload slowdown
LORA_RANK     = 16                   # LoRA rank and alpha
EPOCHS        = 1.0
BATCH_SIZE    = 4                    # 3B is light at seq 1024; drop to 2 if you OOM
GRAD_ACCUM    = 4                    # effective batch = 16
LEARNING_RATE = 2e-4
MAX_ROWS      = 8000                 # 3B's speed affords more data. 0 = all ~20k

USE_DRIVE = True                     # save checkpoints + final model to Google Drive so a
                                     #   Colab disconnect is resumable (just Run-all again)
SAVE_STEPS = 200                     # checkpoint cadence

# Mira's persona, baked in as the DEFAULT system prompt so `ollama run mira` is in
# character for a quick check. main.py overrides this at runtime with her full
# prompt (persona + grounding + memories), so the value here is just for testing.
MIRA_SYSTEM = (
    "You are Mira, a sexy, cute, sarcastic, playful, mischievous anime kitsune "
    "waifu with red hair and fox ears. You tease and banter, never invent facts, "
    "never use emoji or stage directions, and you don't end every line with a "
    "follow-up question. Stay in character; just react and talk like a person."
)

## 2b. Where to save (Google Drive = resumable)

With `USE_DRIVE = True`, checkpoints and the exported model go to
`MyDrive/mira-finetune/`. If Colab disconnects, reconnect and **Run all** again —
training picks up from the latest checkpoint instead of starting over.

In [ ]:
import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/mira-finetune"
else:
    WORK_DIR = "/content/mira-finetune"   # ephemeral: lost if the runtime resets

OUTPUT_DIR = os.path.join(WORK_DIR, "outputs")   # training checkpoints
GGUF_DIR   = os.path.join(WORK_DIR, "mira-gguf") # final gguf + Modelfile
DATA_PATH  = os.path.join(WORK_DIR, "persona_chat.jsonl")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("working dir ->", WORK_DIR)

## 3. Prepare the data

Map each persona-chat row to chat-format messages: the speaker's traits (`persona_b`)
become the system prompt, Persona A → user, Persona B → assistant, and the gold
`reference` is appended as the final assistant turn. (Same logic as the repo's
`prepare_data.py`.)

In [ ]:
import json, re
from datasets import load_dataset

_SPEAKER_RE = re.compile(r"^\s*persona\s*([ab])\s*[:\-]\s*(.*)$", re.IGNORECASE | re.DOTALL)

def persona_system(traits):
    bullets = "\n".join(f"- {t.strip()}" for t in traits if t and t.strip())
    return (
        "You are a character in a casual conversation. Stay fully in character and "
        "reply naturally and conversationally — like a real person, not an assistant. "
        "Keep replies short unless more is genuinely warranted, and do not narrate "
        "actions or add stage directions.\n\nThis is who you are:\n" + bullets
    )

def parse_turn(line):
    m = _SPEAKER_RE.match(line or "")
    if not m:
        return None
    who, text = m.group(1).upper(), m.group(2).strip()
    if not text:
        return None
    return ("user" if who == "A" else "assistant", text)

def build_example(row):
    traits = list(row.get("persona_b") or [])
    if not traits:
        return None
    turns = []
    for line in row.get("dialogue") or []:
        parsed = parse_turn(line)
        if parsed is None:
            continue
        role, text = parsed
        if turns and turns[-1]["role"] == role:
            turns[-1]["content"] += "\n" + text
        else:
            turns.append({"role": role, "content": text})
    if turns and turns[0]["role"] == "assistant":
        turns = turns[1:]
    reference = (row.get("reference") or "").strip()
    if reference:
        if turns and turns[-1]["role"] == "user":
            turns.append({"role": "assistant", "content": reference})
        elif not turns:
            return None
    if len(turns) < 2 or turns[-1]["role"] != "assistant":
        return None
    return {"messages": [{"role": "system", "content": persona_system(traits)}, *turns]}

ds_raw = load_dataset(DATASET, split="train")
if MAX_ROWS and MAX_ROWS < len(ds_raw):
    ds_raw = ds_raw.select(range(MAX_ROWS))

examples = [ex for ex in (build_example(r) for r in ds_raw) if ex is not None]
with open(DATA_PATH, "w", encoding="utf-8") as f:
    for ex in examples:
        f.write(json.dumps(ex, ensure_ascii=False) + "\n")

print(f"prepared {len(examples)} examples ({len(ds_raw) - len(examples)} rows skipped)")
print(json.dumps(examples[0], ensure_ascii=False, indent=2)[:900])

## 4. Load the model + add LoRA

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,  # auto: fp16 on a T4
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0.0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 5. Train (on the assistant replies only)

Resumes automatically from the newest checkpoint in `OUTPUT_DIR` if one exists, so
re-running after a disconnect continues instead of restarting.

In [ ]:
import glob
from datasets import load_dataset as _ld
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

data = _ld("json", data_files=DATA_PATH, split="train")

def format_chat(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
                     for m in batch["messages"]]}

data = data.map(format_chat, batched=True, remove_columns=data.column_names)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=data,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=20,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        save_steps=SAVE_STEPS,
        save_total_limit=3,
        seed=3407,
        bf16=is_bfloat16_supported(),
        fp16=not is_bfloat16_supported(),
        output_dir=OUTPUT_DIR,
        report_to="none",
    ),
)

# Mask the system/user prompt; learn only the assistant turns. These markers are
# Qwen2.5's chat-template delimiters.
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

# Resume from the latest checkpoint if a previous run left one (e.g. after a
# Colab disconnect). Fresh run otherwise.
resume = bool(glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")))
print("resuming from checkpoint" if resume else "starting fresh")
trainer.train(resume_from_checkpoint=resume)

## 6. Quick sanity check (optional)

Generate one reply to eyeball that it's coherent and in-character before exporting.

In [ ]:
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": MIRA_SYSTEM},
    {"role": "user", "content": "ugh, I lost that match again."},
]
inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=120, temperature=0.7, do_sample=True)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))

## 7. Export GGUF + Modelfile

Unsloth merges the LoRA, converts to GGUF via llama.cpp (builds it on first run —
takes a few minutes), and we write a Modelfile with Mira's persona and generation
params matching `prefrontal_cortex.py`.

In [ ]:
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=GGUF_QUANT)

# Newer Unsloth writes the .gguf into a sibling "<GGUF_DIR>_gguf" folder rather than
# GGUF_DIR itself — find wherever it actually landed, and point everything there.
_hits = sorted(glob.glob(os.path.join(GGUF_DIR, "*.gguf")) + glob.glob(GGUF_DIR + "_gguf/*.gguf"))
assert _hits, "no .gguf produced near " + GGUF_DIR
GGUF_DIR = os.path.dirname(_hits[0])
gguf = os.path.basename(_hits[0])

system_escaped = MIRA_SYSTEM.replace('"', '\\"')
modelfile = f'''FROM ./{gguf}

# Generation defaults aligned with prefrontal_cortex.py.
PARAMETER temperature 0.7
PARAMETER num_predict 300
PARAMETER stop "<|im_end|>"

# Default persona for `ollama run {OLLAMA_MODEL_NAME}`. main.py overrides this with
# the full runtime system prompt (persona + grounding + memories) via the chat API.
SYSTEM "{system_escaped}"
'''
with open(os.path.join(GGUF_DIR, "Modelfile"), "w", encoding="utf-8") as f:
    f.write(modelfile)
print("wrote", os.path.join(GGUF_DIR, "Modelfile"), "-> FROM ./" + gguf)

## 8. Get the model onto your PC

If `USE_DRIVE = True`, `mira-gguf/` is **already in your Google Drive** — just
download that folder from Drive on your PC and skip the cell below.

Otherwise, zip + browser-download it here (a 3B q4_k_m GGUF is ~2 GB):

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/mira-gguf", "zip", GGUF_DIR)
files.download(zip_path)

## 9. On your PC

```powershell
# unzip / download mira-gguf, then from inside the folder:
cd mira-gguf
ollama create mira -f Modelfile

# point Mira at the fine-tuned model and run her:
$env:MIRA_MODEL = "mira"
python ..\main.py
```

A/B it against stock Qwen to confirm it helped:
```powershell
ollama run qwen2.5:3b "Reply to: 'ugh I lost again'"
ollama run mira        "Reply to: 'ugh I lost again'"
```
Listen for more natural phrasing and fewer tacked-on "what about you?" endings —
without new invented facts (the grounding rules in Mira's prompt still handle that).